# Introduction to Graph Datbases

The first part of this assignment is designed to give you hands-on experience with graph databases. You will start by setting up an in-memory graph database, for which the support code is already written. Once the database is running, you will execute queries of increasing complexity, exploring how relationships between nodes and edges are stored and retrieved. Through this process, you will gain practical insights into graph database concepts such as connectivity, traversal, and querying using graph-specific languages.

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from utils import setup_database

In [2]:
# Download sample data for the Kuzudb example
data_dir = '../data'
connection = setup_database('../tmp', delete_existing=True)

# Create schema
connection.execute('CREATE NODE TABLE Movie (movieId INT64, year INT64, title STRING, genres STRING, PRIMARY KEY (movieId))')
connection.execute('CREATE NODE TABLE User (userId INT64, PRIMARY KEY (userId))')
connection.execute('CREATE REL TABLE Rating (FROM User TO Movie, rating DOUBLE, timestamp INT64)')
connection.execute('CREATE REL TABLE Tags (FROM User TO Movie, tag STRING, timestamp INT64)')

# Insert data
connection.execute(f'COPY Movie FROM "{data_dir}/movies.csv" (HEADER=TRUE)')
connection.execute(f'COPY User FROM "{data_dir}/users.csv" (HEADER=TRUE)')
connection.execute(f'COPY Rating FROM "{data_dir}/ratings.csv" (HEADER=TRUE)')
connection.execute(f'COPY Tags FROM "{data_dir}/tags.csv" (HEADER=TRUE)')


Loading graph database
Removing existing database at ../tmp


In [3]:
import pandas as pd

## Running Queries

Now that your graph database is set up, you can begin querying it. This section includes seven queries, each increasing in complexity.

In [4]:
# Query 1: Query all nodes with the label 'Movie'. Return those movie nodes. Limit your results to 25
result = connection.execute(""" 
MATCH (m:Movie)
RETURN m
LIMIT 25
""")

df = result.get_as_df()
df.head()

,m
0,"{'_id': {'offset': 0, 'table': 0}, '_label': '..."
1,"{'_id': {'offset': 1, 'table': 0}, '_label': '..."
2,"{'_id': {'offset': 2, 'table': 0}, '_label': '..."
3,"{'_id': {'offset': 3, 'table': 0}, '_label': '..."
4,"{'_id': {'offset': 4, 'table': 0}, '_label': '..."


In [5]:
# Query 2: Query all nodes with the label 'Movie'. Get all connected nodes to the movie nodes. Limit your results to 50
result = connection.execute("""            
MATCH (m:Movie)-[r]-(n)
RETURN m.title, r, n
LIMIT 50
""")

df = result.get_as_df()
df.head()

,m.title,r,n
0,Toy Story (1995),"{'_src': {'offset': 0, 'table': 1}, '_dst': {'...","{'_id': {'offset': 0, 'table': 1}, '_label': '..."
1,Grumpier Old Men (1995),"{'_src': {'offset': 0, 'table': 1}, '_dst': {'...","{'_id': {'offset': 0, 'table': 1}, '_label': '..."
2,Heat (1995),"{'_src': {'offset': 0, 'table': 1}, '_dst': {'...","{'_id': {'offset': 0, 'table': 1}, '_label': '..."
3,Seven (a.k.a. Se7en) (1995),"{'_src': {'offset': 0, 'table': 1}, '_dst': {'...","{'_id': {'offset': 0, 'table': 1}, '_label': '..."
4,"Usual Suspects, The (1995)","{'_src': {'offset': 0, 'table': 1}, '_dst': {'...","{'_id': {'offset': 0, 'table': 1}, '_label': '..."


In [6]:
# Query 3: Count the total number of nodes in the database
# Hint: Use the `COUNT` function to count the number of nodes
result = connection.execute("""
MATCH (n)
RETURN COUNT(n) AS total_nodes
""")

df = result.get_as_df()
df.head()

,total_nodes
0,10352


In [8]:
# Query 4: Query all nodes with the label 'User'. Count the degree for these nodes. Filter the nodes where the user rated more than 3 movies. Return the users and the degree
# Hint: First find all users and their ratings, then count the degree, and finally filter the results to only include users with more than 3 ratings
result = connection.execute("""MATCH (u:User)-[r]->(m:Movie)
WITH u, COUNT(r) AS degree
WHERE degree > 3
RETURN u, degree
ORDER BY degree""")

df = result.get_as_df()
df.head()

,u,degree
0,"{'_id': {'offset': 52, 'table': 1}, '_label': ...",20
1,"{'_id': {'offset': 575, 'table': 1}, '_label':...",20
2,"{'_id': {'offset': 193, 'table': 1}, '_label':...",20
3,"{'_id': {'offset': 405, 'table': 1}, '_label':...",20
4,"{'_id': {'offset': 146, 'table': 1}, '_label':...",20


In [10]:
# Query 5: Query all nodes with the label 'Movie'. Each node has a 'genre' attribute. Count the number of nodes per genre
# Hint: Use the `WITH` clause to group by genres and count the number of movies
result = connection.execute("""MATCH (m:Movie)
WHERE m.genres IS NOT NULL
WITH m.genres AS genre, COUNT(m) AS count
RETURN genre, count
ORDER BY count DESC""")

df = result.get_as_df()
df.head()

,genre,count
0,Drama,1058
1,Comedy,950
2,Comedy|Drama,435
3,Comedy|Romance,363
4,Drama|Romance,349


In [11]:
# Query 6: Query all nodes with the label 'Movie' and 'User', and the edge 'Rating' between movie and user. Each edge 'Rating' has a rating. Find the top 10 rated movies by average rating score
# Hint: Use the AVG clause to calculate an average. Use the `ORDER BY` clause to sort the movies by rating in descending order
result = connection.execute("""
MATCH (u:User)-[r:Rating]->(m:Movie)
WITH m, AVG(r.rating) AS avg_rating
RETURN m.title AS movie, avg_rating
ORDER BY avg_rating DESC
LIMIT 10
""")

df = result.get_as_df()
df.head()

,movie,avg_rating
0,Moscow Does Not Believe in Tears (Moskva sleza...,5.0
1,"Woman Is a Woman, A (femme est une femme, Une)...",5.0
2,"Marriage of Maria Braun, The (Ehe der Maria Br...",5.0
3,Sun Alley (Sonnenallee) (1999),5.0
4,Tom and Jerry: A Nutcracker Tale (2007),5.0


In [12]:
# Query 7: Query all nodes with the label 'Movie' and 'User', and the edge 'Rating' between movie and user. Find pairs of movies often rated by the same users
result = connection.execute("""
MATCH (u:User)-[:Rating]->(m1:Movie),
      (u)-[:Rating]->(m2:Movie)
WHERE m1.title < m2.title
WITH m1.title AS movie1, m2.title AS movie2, COUNT(u) AS shared_users
RETURN movie1, movie2, shared_users
ORDER BY shared_users DESC
LIMIT 50
""")

df = result.get_as_df()
df.head()

,movie1,movie2,shared_users
0,Forrest Gump (1994),"Shawshank Redemption, The (1994)",231
1,Forrest Gump (1994),Pulp Fiction (1994),230
2,Pulp Fiction (1994),"Shawshank Redemption, The (1994)",222
3,Pulp Fiction (1994),"Silence of the Lambs, The (1991)",207
4,Forrest Gump (1994),"Silence of the Lambs, The (1991)",199
